##1.로지스틱 회귀분석 예제 :종속변수가 범주형일 때 사용하는 회귀분석



####1)고객이탈지수(churn)를 종속변수로 하여 로지스틱회귀분석 수행
     고객 이탈 여부(0:유지, 1:이탈)
####2) 데이터에서 상수항을 추가한 로지스틱 회귀분석 후 유의하지 않은 변수의 개수 구하기
####3)유의미한 변수만을 독립변수로 하여, 로지스틱 회귀 재 수행한 후 회귀계수의 평균 구하기

In [ ]:
!git clone 'https://'github.com/KangKyungHwa/DSMH_2/''

# 1. 나눔고딕 폰트 설치
!sudo apt-get update -qq
!sudo apt-get install -y fonts-nanum-extra -qq

# 2. 시스템 폰트 캐시 업데이트
!sudo fc-cache -fv

# 3. Matplotlib 폰트 캐시 삭제
!rm -rf ~/.cache/matplotlib

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.size'] = 11

Cloning into 'DSMH_2'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 188 (delta 4), reused 2 (delta 2), pack-reused 179 (from 1)
Receiving objects: 100% (188/188), 1.71 MiB | 4.00 MiB/s, done.
Resolving deltas: 100% (67/67), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unse

##2. 데이터 수집 및 전처리

* (1) 데이터 수집

In [7]:
from os import setns
import pandas as pd # 데이터 로드 및 전처리에 사용
import numpy as np
import statsmodels.api as sm # 통계적 회귀 모델 생성 및 분석에 사용

df = pd.read_csv("DSMH_2/__10_22/customer_churn.csv")

print(df.head(3), '\n\n')
print(df.tail(3))

   age  calls  sms_usage  gender  monthly_charge  churn  contract_length  \
0   56     25          3       1       70.011145      0               21   
1   69     27         34       0       73.885745      0                6   
2   46     46        191       0       54.144566      0                6   

   data_usage  internet_usage  customer_support_calls  \
0    2.621644               6                       7   
1   19.940611               7                       8   
2   10.541705               6                       9   

   payment_method_Credit Card  payment_method_PayPal  region_North  \
0                           0                      1             0   
1                           0                      0             0   
2                           0                      0             1   

   region_South  region_West  
0             0            0  
1             0            1  
2             0            0   


     age  calls  sms_usage  gender  monthly_charge  churn 

* (1) 데이터 정보확인

In [8]:
print (df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   age                         200 non-null    int64  
 1   calls                       200 non-null    int64  
 2   sms_usage                   200 non-null    int64  
 3   gender                      200 non-null    int64  
 4   monthly_charge              200 non-null    float64
 5   churn                       200 non-null    int64  
 6   contract_length             200 non-null    int64  
 7   data_usage                  200 non-null    float64
 8   internet_usage              200 non-null    int64  
 9   customer_support_calls      200 non-null    int64  
 10  payment_method_Credit Card  200 non-null    int64  
 11  payment_method_PayPal       200 non-null    int64  
 12  region_North                200 non-null    int64  
 13  region_South                200 non

* (2) 로지스틱 회귀수행

In [27]:
#독립변수와(X) 종속변수(y) 설정

X=df.drop('churn', axis=1)
y =df['churn']

# 상수항추가

X= sm.add_constant(X)

#로지스틱 회귀모델 학습
model = sm.Logit(y, X).fit()

#회귀분석 결과 출력
model_summary = model.summary()
p_values = model.pvalues

Optimization terminated successfully.
         Current function value: 0.090695
         Iterations 11


3. 유의하지 않은 변수 갯수

In [30]:
# 유의하지 않은 변수 나타내기

non_significant_vars=p_values[p_values>0.05].count()
print(model_summary)
print(f"유의하지 않은 변수의 개수: {non_significant_vars}")


                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  200
Model:                          Logit   Df Residuals:                      185
Method:                           MLE   Df Model:                           14
Date:                Wed, 21 Oct 2026   Pseudo R-squ.:                  0.6747
Time:                        18:38:40   Log-Likelihood:                -18.139
converged:                       True   LL-Null:                       -55.754
Covariance Type:            nonrobust   LLR p-value:                 2.148e-10
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                        -48.4932     13.372     -3.626      0.000     -74.702     -22.285
age                            0.3010      0.086      3.490      0.000       0.132 

4. 유의미한 변수만을 독립변수로 하여 로지스틱 회귀모델을 다시 만든다

In [34]:
significant_vars = p_values[p_values <= 0.05]
print(significant_vars)


const             0.000287
age               0.000484
calls             0.002091
monthly_charge    0.000821
dtype: float64


In [35]:
#유의미한 변수만을 사용하여 로지스틱 회귀 수행
X_significant = df[['age','calls','monthly_charge']] #유의미한 변수만 선택
X_significant= sm.add_constant(X_significant)

#로지스틱 회귀모델 재 학습
model_significant = sm.Logit(y, X_significant).fit()

#회귀계수의 평균계산
coeff_mean = model_significant.params.mean()

#회귀계수의 평균 출력
print(model_significant.summary())
print("회귀계수의 평균:", coeff_mean)


Optimization terminated successfully.
         Current function value: 0.128481
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:                  churn   No. Observations:                  200
Model:                          Logit   Df Residuals:                      196
Method:                           MLE   Df Model:                            3
Date:                Wed, 21 Oct 2026   Pseudo R-squ.:                  0.5391
Time:                        18:51:59   Log-Likelihood:                -25.696
converged:                       True   LL-Null:                       -55.754
Covariance Type:            nonrobust   LLR p-value:                 5.554e-13
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const            -40.2967      9.192     -4.384      0.000     -58.312     -22.281
age             

###해석 : 로지스틱 회귀분석에서는 종속변수가 확률(0과 1사이) 이기 때문에 해석이 달라진다.

다른 변수들이 일정할 때 age 가 1단위 증가하면 이탈할 로그 오즈(Log Odds : 이탈할 확률 대비 유지할 확률의 로그값) 값이 0.2045만큼 증가한다는 의미
###즉, 다른 조건이 동일할 때 나이가 1살 많아질수록 이탈(churn=1)할 확률이 이탈하지 않을 확률에 비해 약 1.227배(약 22.7% 증가) 더 높아진다는 것을 의미합니다.